# Контроль качества данных эксперимента 3

**Статус:** техническая инвентаризация и диагностический ноутбук
реокардиомонитора РНЦХ. Он не рассчитывает функции сердца.

Паспорт серии вынесен в 10.10_Паспорт_эксперимента_3.md. Дата проверяется по
первичному протоколу. Ориентировочные дыхательные интервалы в сохранённых
графиках не являются количественной разметкой. Постороннее исследование на
другом приборе исключено из доказательной цепочки.

Сохранённые выводы ниже относятся к историческому запуску до миграции и должны
быть воспроизведены после подключения внешней конфигурации.


In [ ]:
# Импорты и внешняя конфигурация
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

CONFIG_ENV = "KALMYKOV_EXP03_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp03_paths.example.json"
    )

CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
CSV_ROOT = (
    Path(CONFIG["data_root"]).expanduser().resolve()
    / CONFIG["csv_subdir"]
)

plt.rcParams.update({
    "figure.figsize": (15, 9),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 13,
})

CHANNEL_COLUMNS = {
    1: {"rheo": 1, "base": 2, "qs": 3},
    2: {"rheo": 5, "base": 6, "qs": 7},
}

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_record(path):
    frame = pd.read_csv(path)
    frame.columns = [
        "time_s",
        "rheo_1_mohm",
        "base_1_ohm",
        "qs_1_ohm",
        "ecg_v",
        "rheo_2_mohm",
        "base_2_ohm",
        "qs_2_ohm",
    ]
    return frame


## 1. Инвентаризация и соответствие файлов

Файлы с одинаковым SHA-256 считаются физическими копиями одной записи. Для анализа выбирается каноническая копия с самым коротким относительным путём.


In [ ]:
inventory_rows = []
for path in sorted(CSV_ROOT.rglob("*.csv")):
    data = read_record(path)
    dt = float(np.median(np.diff(data["time_s"])))
    inventory_rows.append({
        "path": path,
        "relative_path": str(path.relative_to(CSV_ROOT)),
        "file": path.name,
        "time": path.name[11:19].replace("-", ":"),
        "sha256": file_sha256(path),
        "rows": len(data),
        "duration_s": float(data["time_s"].iloc[-1]),
        "fs_hz": 1.0 / dt,
        "path_depth": len(path.relative_to(CSV_ROOT).parts),
    })

inventory = pd.DataFrame(inventory_rows)
inventory["copies"] = inventory.groupby("sha256")["sha256"].transform("size")
canonical = (inventory.sort_values(["sha256", "path_depth", "relative_path"])
             .drop_duplicates("sha256")
             .sort_values("time")
             .reset_index(drop=True))

ASSIGNMENTS = {
    "13:39:22": ("Настройка, короткий фрагмент", "не входит", "высокая", "6.2 с; только BASE1; QS обоих каналов на потолке 4700"),
    "13:39:32": ("Предварительная запись канала 1", "не входит", "средняя", "107.4 с; только BASE1≈73 Ом; в протоколе не упомянута"),
    "13:57:33": ("Настройка перед основными пробами", "не входит", "средняя", "34.4 с; BASE1≈61 Ом, BASE2≈0; в протоколе не упомянута"),
    "14:07:01": ("ТТРКГ, канал 1 отдельно", "описан", "высокая", "время и длительность совпадают; BASE2=0"),
    "14:09:42": ("Боковой канал 2 отдельно", "описан", "высокая", "время и длительность совпадают; BASE1=0"),
    "14:17:16": ("Поочерёдное отключение каналов", "описан как 14:08", "высокая", "видна последовательность оба→2→оба→1→оба"),
    "14:19:52": ("Оба канала: дыхательный протокол", "описан как 14:17", "высокая", "73.6 с и структура 30 с дыхания→задержка→3 цикла→задержка"),
}

canonical[["assignment", "protocol", "confidence", "evidence"]] = canonical["time"].apply(
    lambda value: pd.Series(ASSIGNMENTS.get(value, ("Не определено", "нет", "низкая", "")))
)

mapping_table = canonical[["time", "file", "duration_s", "fs_hz", "copies",
                           "assignment", "protocol", "confidence", "evidence"]].copy()
mapping_table["duration_s"] = mapping_table["duration_s"].round(2)
mapping_table["fs_hz"] = mapping_table["fs_hz"].round(1)
mapping_table.columns = ["Время", "CSV", "Длительность, с", "Частота, Гц", "Копий",
                         "Назначение", "Связь с протоколом", "Уверенность", "Основание"]
display(mapping_table.style.hide(axis="index").set_properties(**{"text-align": "left"}))


### Главный результат сопоставления

- `14-07-01` — отдельный канал 1, ТТРКГ.
- `14-09-42` — отдельный канал 2, боковая сборка.
- `14-17-16` — тест отключения каналов, хотя в тексте протокола для него указано 14:08.
- `14-19-52` — совместный дыхательный протокол, хотя в тексте указано 14:17.
- Последние четыре CSV продублированы байт-в-байт в корне `TTRKG_and_SIDE` и в подпапке `2026-07-15`; на графиках каждая запись показана один раз.

Таким образом, наиболее вероятна не потеря файлов, а сдвиг времён двух последних пунктов в рукописном описании протокола.


In [ ]:
canonical_by_time = dict(zip(canonical["time"], canonical["path"]))

def channel_stats(path, channels):
    data = read_record(path)
    result = []
    for channel in channels:
        base = data[f"base_{channel}_ohm"]
        qs = data[f"qs_{channel}_ohm"]
        rheo = data[f"rheo_{channel}_mohm"]
        result.append({
            "Канал": channel,
            "BASE median, Ом": base.median(),
            "BASE p01–p99, Ом": f"{base.quantile(0.01):.2f}–{base.quantile(0.99):.2f}",
            "QS median, Ом": qs.median(),
            "QS=4700, %": 100 * np.mean(qs >= 4699),
            "RHEO p01–p99, мОм": f"{rheo.quantile(0.01):.1f}–{rheo.quantile(0.99):.1f}",
        })
    table = pd.DataFrame(result)
    for col in ["BASE median, Ом", "QS median, Ом", "QS=4700, %"]:
        table[col] = table[col].round(2)
    return table

STAGE_COLORS = {
    "Свободное дыхание": "#dbeafe",
    "Вдох + задержка": "#dcfce7",
    "Форсированное дыхание": "#fee2e2",
    "Задержка на выдохе": "#fef3c7",
}

def plot_record(path, title, channels, stages=None):
    data = read_record(path)
    time = data["time_s"]
    fs = 1.0 / np.median(np.diff(time))
    window = int(round(fs))
    if window % 2 == 0:
        window += 1

    fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True,
                             gridspec_kw={"height_ratios": [2.2, 1, 1, 1]})
    colors = {1: "#2563eb", 2: "#dc2626"}

    for channel in channels:
        rheo = data[f"rheo_{channel}_mohm"]
        smooth = rheo.rolling(window, center=True, min_periods=1).median()
        axes[0].plot(time, rheo, color=colors[channel], alpha=0.16, linewidth=0.5)
        axes[0].plot(time, smooth, color=colors[channel], linewidth=1.7,
                     label=f"канал {channel}, медиана 1 с")
        axes[1].plot(time, data[f"base_{channel}_ohm"], color=colors[channel],
                     linewidth=1.2, label=f"BASE {channel}")
        axes[2].plot(time, data[f"qs_{channel}_ohm"], color=colors[channel],
                     linewidth=1.2, label=f"QS {channel}")

    axes[3].plot(time, data["ecg_v"], color="#111827", linewidth=0.7, label="ЭКГ")
    axes[0].set_ylabel("RHEO, мОм")
    axes[1].set_ylabel("BASE, Ом")
    axes[2].set_ylabel("QS, Ом")
    axes[3].set_ylabel("ЭКГ, В")
    axes[3].set_xlabel("Время от начала CSV, с")
    axes[0].set_title(title)

    if stages:
        for start, end, label in stages:
            color = STAGE_COLORS.get(label, "#e5e7eb")
            for axis in axes:
                axis.axvspan(start, end, color=color, alpha=0.26, linewidth=0)
            axes[0].text((start + end) / 2, 0.98, label, ha="center", va="top",
                         fontsize=9, transform=axes[0].get_xaxis_transform())

    for axis in axes:
        axis.legend(loc="upper right", ncol=max(1, len(channels)))
        axis.margins(x=0)
    fig.tight_layout()
    plt.show()

def active_intervals(path, threshold_ohm=5.0):
    data = read_record(path)
    time = data["time_s"].to_numpy()
    base_1 = data["base_1_ohm"].to_numpy()
    base_2 = data["base_2_ohm"].to_numpy()
    state = (base_1 > threshold_ohm).astype(int) + 2 * (base_2 > threshold_ohm).astype(int)
    cuts = np.r_[0, np.flatnonzero(state[1:] != state[:-1]) + 1, len(state)]
    names = {0: "ни один", 1: "только 1", 2: "только 2", 3: "оба"}
    rows = []
    for left, right in zip(cuts[:-1], cuts[1:]):
        if time[right - 1] - time[left] < 0.25:
            continue
        rows.append({
            "Начало, с": time[left],
            "Конец, с": time[right - 1],
            "Активны": names[int(state[left])],
            "BASE1 median, Ом": np.median(base_1[left:right]),
            "BASE2 median, Ом": np.median(base_2[left:right]),
            "QS1 median, Ом": np.median(data["qs_1_ohm"].iloc[left:right]),
            "QS2 median, Ом": np.median(data["qs_2_ohm"].iloc[left:right]),
        })
    return pd.DataFrame(rows).round(2)


## 2. ТТРКГ, канал 1 отдельно — `14-07-01`

По протоколу: свободное дыхание 30 с, вдох и задержка около 15 с, форсированное дыхание, задержка на выдохе. Границы после 30-й секунды отмечены ориентировочно.


In [ ]:
path_1407 = canonical_by_time["14:07:01"]
stages_1407 = [
    (0, 30, "Свободное дыхание"),
    (30, 45, "Вдох + задержка"),
    (45, 53, "Форсированное дыхание"),
    (53, 67.42, "Задержка на выдохе"),
]
display(channel_stats(path_1407, [1]).style.hide(axis="index"))
plot_record(path_1407, "14:07 — ТТРКГ, канал 1 отдельно", [1], stages_1407)


**Наблюдение:** `BASE1≈101 Ом`, а не около 80 Ом, указанных в примечании протокола. `QS1=4700 Ом` на всей записи — это тот же потолок, который наблюдается у отключённых каналов; поэтому запись нужно считать потенциально перегруженной/некорректно согласованной, пока не будет расшифрована шкала QS прибора.


## 3. Боковая сборка, канал 2 отдельно — `14-09-42`

По протоколу: свободное дыхание 30 с, задержка на вдохе около 15 с, три форсированных цикла, задержка на выдохе. Три крупных цикла хорошо видны около 49–59 с.


In [ ]:
path_1409 = canonical_by_time["14:09:42"]
stages_1409 = [
    (0, 30, "Свободное дыхание"),
    (30, 49, "Вдох + задержка"),
    (49, 59, "Форсированное дыхание"),
    (59, 69.22, "Задержка на выдохе"),
]
display(channel_stats(path_1409, [2]).style.hide(axis="index"))
plot_record(path_1409, "14:09 — боковая сборка 140/70 мм, канал 2 отдельно", [2], stages_1409)


**Наблюдение:** `BASE2≈37.2 Ом` согласуется с указанными в протоколе 36 Ом. `QS2≈343 Ом` стабилен и не достигает потолка 4700. Канал 1 в этой записи действительно отключён (`BASE1=0`).


## 4. Поочерёдное отключение каналов — фактически `14-17-16`

Состояния определены автоматически по условию `BASE>5 Ом`. Переходы полностью совпадают с описанным тестом: сначала оба канала, затем отключён первый, снова оба, затем отключён второй, в конце снова оба.


In [ ]:
path_1417 = canonical_by_time["14:17:16"]
switch_intervals = active_intervals(path_1417)
display(switch_intervals.style.hide(axis="index"))
switch_stages = [(row["Начало, с"], row["Конец, с"], row["Активны"])
                 for _, row in switch_intervals.iterrows()]
plot_record(path_1417, "14:17 — поочерёдное отключение каналов", [1, 2], switch_stages)


**Наблюдение:** при совместной работе `BASE1≈54 Ом`, `BASE2≈41 Ом`; при отключении первого `BASE1=0`, а второй остаётся около 37 Ом; при отключении второго `BASE2=0`, а первый возрастает примерно до 93 Ом. Это количественно подтверждает взаимное влияние каналов. Значение `QS=4700` появляется у отключённого канала и у канала 1 при его одиночной работе, поэтому его следует рассматривать как индикатор предельного/нештатного состояния.


## 5. Оба канала: дыхательный протокол — фактически `14-19-52`

Структура записи соответствует протоколу: около 30 с свободного дыхания, вдох и задержка, три форсированных цикла, затем задержка на выдохе.


In [ ]:
path_1419 = canonical_by_time["14:19:52"]
stages_1419 = [
    (0, 30, "Свободное дыхание"),
    (30, 46, "Вдох + задержка"),
    (46, 57, "Форсированное дыхание"),
    (57, 73.62, "Задержка на выдохе"),
]
display(channel_stats(path_1419, [1, 2]).style.hide(axis="index"))
plot_record(path_1419, "14:19 — оба канала, дыхательный протокол", [1, 2], stages_1419)


**Наблюдение:** при совместной работе медианы составляют примерно `BASE1=54.4 Ом` и `BASE2=40.7 Ом`; второй уровень хорошо совпадает с протоколом, первый ниже ожидаемых 60 Ом, но близок по порядку. `QS1≈1307 Ом`, `QS2≈368 Ом`, без выхода на потолок 4700. Оба RHEO-сигнала синхронно отражают дыхательные манёвры, но на форсированных вдохах/выдохах достигают ограничения диапазона.


## 6. Выводы первой итерации

1. Четыре протокольные записи идентифицированы с высокой уверенностью; два времени в текстовом протоколе, вероятно, записаны неточно.
2. Анализировать следует уникальные SHA-256, иначе четыре основные записи будут посчитаны дважды.
3. Наиболее чистая одиночная запись — боковой канал `14-09-42`: базовый импеданс согласуется с протоколом, QS стабилен.
4. Одиночная запись ТТРКГ `14-07-01` требует осторожности: BASE выше ожидаемого, QS находится на потолке.
5. Тест `14-17-16` доказывает взаимное влияние каналов: базовые уровни существенно меняются при отключении соседнего канала.
6. Для следующей итерации нужны расшифровка физического смысла и допустимых диапазонов QS, а также подтверждение границ дыхательных этапов по журналу/видео или ручным меткам оператора.
